In [1]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from aeon.datasets import *
import numpy as np
from aeon.transformations.collection.convolution_based import MultiRocket
from aeon.transformations.collection.convolution_based._hydra import HydraTransformer
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score
from tscglue.models import NoScaler
from aeon.transformations.collection.unequal_length import Padder
from time import perf_counter

### MultiRocket features

In [2]:
def mr_fit(x_train, x_test, rand):
    mr = MultiRocket(random_state=rand)
    X_train_mr = mr.fit_transform(x_train)
    X_test_mr = mr.transform(x_test)
    return X_train_mr, X_test_mr

### Hydra features

In [3]:
def hyd_fit(x_train, x_test, rand):
    hyd = HydraTransformer(random_state=rand)
    X_train_hy = hyd.fit_transform(x_train)
    X_test_hy = hyd.transform(x_test)
    return X_train_hy, X_test_hy

### Applying scalers and concatenate features


In [4]:
def apply_scaler_fit(scaler_hy, scaler_mr, x_train, x_test, rand):
    X_train_hy, X_test_hy = hyd_fit(x_train, x_test, rand)
    X_train_mr, X_test_mr = mr_fit(x_train, x_test, rand)

    X_train_hy_scaled = scaler_hy.fit_transform(X_train_hy)
    X_test_hy_scaled = scaler_hy.transform(X_test_hy)
    X_train_mr_scaled = scaler_mr.fit_transform(X_train_mr)
    X_test_mr_scaled = scaler_mr.transform(X_test_mr)

    X_train_f = np.concatenate([X_train_mr_scaled, X_train_hy_scaled], axis=1)
    X_test_f  = np.concatenate([X_test_mr_scaled, X_test_hy_scaled], axis=1)

    return X_train_f, X_test_f

In [5]:
 def get_acc_matrix(x_train, x_test, y_train, y_test):
    scalers = [
        MinMaxScaler(),
        NoScaler(),
        StandardScaler(),
        RobustScaler()
    ]
    n = len(scalers)

    acc_matrix = np.zeros((n, n))


    clf = RidgeClassifierCV()

    for i in range(n):
        for j in range(n):
            x_train_scaled, x_test_scaled = apply_scaler_fit(scalers[i], scalers[j], x_train, x_test, 42)
            clf.fit(x_train_scaled, y_train)
            preds = clf.predict(x_test_scaled)
            acc_matrix[i,j] = accuracy_score(y_test, preds)

    return acc_matrix

In [6]:
def pad_train_test(x_train, x_test):
    padder = Padder()
    x_all = list(x_train) + list(x_test)
    x_all_padded = padder.fit_transform(x_all)

    x_train_p = x_all_padded[:len(x_train)]
    x_test_p  = x_all_padded[len(x_train):]
    return x_train_p, x_test_p

In [8]:
matrixes = []

start_time = perf_counter()
x_train, y_train = load_arrow_head(split="train")
x_test, y_test = load_arrow_head(split="test")
x_train_p, x_test_p = pad_train_test(x_train, x_test)
matrixes.append(get_acc_matrix(x_train_p, x_test_p, y_train, y_test))
print("Finished evaluating arrow head dataset in " + str(perf_counter() - start_time) + " seconds.")

start_time = perf_counter()
x_train, y_train = load_osuleaf(split="train")
x_test, y_test = load_osuleaf(split="test")
x_train_p, x_test_p = pad_train_test(x_train, x_test)
matrixes.append(get_acc_matrix(x_train_p, x_test_p, y_train, y_test))
print("Finished evaluating OSULeaf dataset in " + str(perf_counter() - start_time) + " seconds.")

start_time = perf_counter()
x_train, y_train = load_italy_power_demand(split="train")
x_test, y_test = load_italy_power_demand(split="test")
x_train_p, x_test_p = pad_train_test(x_train, x_test)
matrixes.append(get_acc_matrix(x_train_p, x_test_p, y_train, y_test))
print("Finished evaluating Italy power demand dataset in " + str(perf_counter() - start_time) + " seconds.")

start_time = perf_counter()
x_train, y_train = load_japanese_vowels(split="train")
x_test, y_test = load_japanese_vowels(split="test")
x_train_p, x_test_p = pad_train_test(x_train, x_test)
matrixes.append(get_acc_matrix(x_train_p, x_test_p, y_train, y_test))
print("Finished evaluating Japanese vowels dataset in " + str(perf_counter() - start_time) + " seconds.")

start_time = perf_counter()
x_train, y_train = load_acsf1(split="train")
x_test, y_test = load_acsf1(split="test")
x_train_p, x_test_p = pad_train_test(x_train, x_test)
matrixes.append(get_acc_matrix(x_train_p, x_test_p, y_train, y_test))
print("Finished evaluating acsf1 dataset in " + str(perf_counter() - start_time) + " seconds.")


Finished evaluating arrow head dataset in 84.21752058097627 seconds.
Finished evaluating OSULeaf dataset in 271.8585705489968 seconds.
Finished evaluating Italy power demand dataset in 61.43294511898421 seconds.
Finished evaluating Japanese vowels dataset in 75.37582222203491 seconds.
Finished evaluating acsf1 dataset in 418.3347254759865 seconds.


In [9]:
avg_matrix = np.mean(matrixes, axis=0)
print(avg_matrix)

[[0.94745583 0.92962135 0.94031297 0.78024395]
 [0.92637288 0.92873351 0.92999582 0.87081189]
 [0.93889439 0.92962135 0.93517011 0.79616092]
 [0.92101663 0.93142699 0.92354718 0.78720243]]


In [10]:
scaler_names = ["minmax", "none", "standard", "robust"]

row_labels = [f"HY_{s}" for s in scaler_names]
col_labels = [f"MR_{s}" for s in scaler_names]
print("Average accuracy over 5 datasets:")
header = " " * 12
for col in col_labels:
    header += f"{col:>12}"
print(header)

for i, row_label in enumerate(row_labels):
    row_str = f"{row_label:<12}"
    for j in range(len(col_labels)):
        row_str += f"{avg_matrix[i, j]:12.3f}"
    print(row_str)

Average accuracy over 5 datasets:
               MR_minmax     MR_none MR_standard   MR_robust
HY_minmax          0.947       0.930       0.940       0.780
HY_none            0.926       0.929       0.930       0.871
HY_standard        0.939       0.930       0.935       0.796
HY_robust          0.921       0.931       0.924       0.787


Using MinMax scaler for both Hydra and Multirocket features gives the highest average accuracy across the 5 evaluated datasets. Applying Robust scaling on MultiRocket features consistantly produces the worst performance and should be avoided.